### Dependencies

In [36]:
import anthropic
# best structured output library
import instructor

from qdrant_client import QdrantClient
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import os
import json

load_dotenv("../../.env")

True

In [44]:
# print in pretty
def pprint(response):
        print(json.dumps(response.model_dump(), indent=2), '\n')

### Mock Example

In [28]:
PROMPT = """
You are a helpful assistant
Return an answer to the question
Question: What is your name? Keep it short
"""

In [38]:
import json

client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=8096,
#     system=PROMPT,
    messages=[
        {"role": "user", "content": PROMPT}
    ],
    temperature=0
)
pprint(response)

{
  "id": "msg_01T1yPyneYVBnxecEemLCi3G",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "I'm Claude, an AI assistant made by Anthropic.",
      "type": "text"
    }
  ],
  "model": "claude-haiku-4-5-20251001",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "not_available",
    "input_tokens": 32,
    "output_tokens": 16,
    "server_tool_use": null,
    "service_tier": "standard"
  }
}


### Add Instructor (Structured Outputs)

In [30]:
instructor_client = instructor.from_anthropic(anthropic.Anthropic())

In [31]:
# Create pydantic model for Structured Output
class RAGGenerationResponse(BaseModel):
        answer: str = Field(description="The answer to the question")

In [39]:
response = instructor_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=8096,
#     system=PROMPT,
    messages=[
        {"role": "user", "content": PROMPT}
    ],
    temperature=0,
    response_model=RAGGenerationResponse,
)
pprint(response)

{
  "answer": "I'm Claude, an AI assistant made by Anthropic."
}


In [45]:
response, raw_output = instructor_client.messages.create_with_completion(
    model="claude-haiku-4-5-20251001",
    max_tokens=8096,
#     system=PROMPT,
    messages=[
        {"role": "user", "content": PROMPT}
    ],
    temperature=0,
    response_model=RAGGenerationResponse,
) 
pprint(response)
pprint(raw_output)

{
  "answer": "I'm Claude, an AI assistant made by Anthropic."
} 

{
  "id": "msg_01EEXKNFeZaMMYunF7yNBsfr",
  "container": null,
  "content": [
    {
      "id": "toolu_01SmYsfzLxfi8RTnhtZ2bKam",
      "caller": {
        "type": "direct"
      },
      "input": {
        "answer": "I'm Claude, an AI assistant made by Anthropic."
      },
      "name": "RAGGenerationResponse",
      "type": "tool_use"
    }
  ],
  "model": "claude-haiku-4-5-20251001",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "tool_use",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "not_available",
    "input_tokens": 721,
    "output_tokens": 44,
    "server_tool_use": null,
    "service_tier": "standard"
  }
} 



In [46]:
# Create pydantic model for Structured Output
class RAGGenerationResponse(BaseModel):
        answer: str = Field(description="The answer to the question")
        reasoning: str = Field(description="The reasoning to the answer")

In [47]:
# see the response contains the description as well even though we did not change anything in the prompt
# instructor injects the pydantic class and its description with additinal instuction to give the required output

response, raw_output = instructor_client.messages.create_with_completion(
    model="claude-haiku-4-5-20251001",
    max_tokens=8096,
#     system=PROMPT,
    messages=[
        {"role": "user", "content": PROMPT}
    ],
    temperature=0,
    response_model=RAGGenerationResponse,
) 
pprint(response)
pprint(raw_output)

{
  "answer": "Claude",
  "reasoning": "The user is asking for my name and requested a short answer. My name is Claude, which is a concise response to their question."
} 

{
  "id": "msg_01VbRSs23mzofeuMGuHgekeN",
  "container": null,
  "content": [
    {
      "id": "toolu_01NfiQPtrVfep6XWF5Cmuwco",
      "caller": {
        "type": "direct"
      },
      "input": {
        "answer": "Claude",
        "reasoning": "The user is asking for my name and requested a short answer. My name is Claude, which is a concise response to their question."
      },
      "name": "RAGGenerationResponse",
      "type": "tool_use"
    }
  ],
  "model": "claude-haiku-4-5-20251001",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "tool_use",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference

### RAG example

In [ ]:
class RAGGenerationResponse(BaseModel):
        answer: str = Field(description="The answer to the question")


In [50]:
from dotenv import load_dotenv
import os
import voyageai

from qdrant_client import QdrantClient
import anthropic

def get_embedding(voyageai_client, text, model = 'voyage-3'):
        result = voyageai_client.embed(
                [text],
                model=model,
                input_type="document"
        )
        return result.embeddings[0]


def retrieve_data(voyageai_client, query, qdrant_client, k=5):
        query_embedding = get_embedding(voyageai_client, query)
        results = qdrant_client.query_points(
                collection_name="Amazon-items-collection-00",
                query=query_embedding,
                limit=k,
        )

        retrieved_context_ids = []
        retrieved_context = []
        similarity_scores = []
        retrieved_context_ratings = []

        for item in results.points:
                retrieved_context_ids.append(item.payload['parent_asin'])
                retrieved_context.append(item.payload['description'])
                retrieved_context_ratings.append(item.payload['average_rating'])
                similarity_scores.append(item.score)
        
        return {
                "retrieved_context_ids": retrieved_context_ids,
                "retrieved_context": retrieved_context,
                "retrieved_context_ratings": retrieved_context_ratings,
                "similarity_scores": similarity_scores,
        }


def process_context(context):
        formatted_context = ""
        for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrieved_context'], context['retrieved_context_ratings']):
                formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
        return formatted_context


def build_pompt(preprocessed_context, question):
        prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context

Instrctions:
- You need to answer the question based on the provided context only
- Never use word context and refer to it as the available products

Context:
{preprocessed_context}

Question:
{question}
        """
        return prompt


def generate_answer(prompt):
        message, raw_response = instructor_client.messages.create_with_completion(
                max_tokens=2000,
                messages=[
                        {
                        "role": "user",
                        "content": prompt,
                        }
                ],
                model="claude-haiku-4-5",
                temperature=0,
                response_model=RAGGenerationResponse,
        )
        return message


def rag_pipeline(question, top_k=10):
        load_dotenv()
        VOYAGE_API_KEY = os.environ.get("VOYAGE_API_KEY")
        voyageai_client = voyageai.Client(api_key=VOYAGE_API_KEY)
        qdrant_client = QdrantClient(url='http://localhost:6333')
        retrieved_context = retrieve_data(voyageai_client, question, qdrant_client, top_k)
        preprocessed_context = process_context(retrieved_context)
        prompt = build_pompt(preprocessed_context, question)
        answer = generate_answer(prompt)

        # for evaluation we should return the following
        final_result = {
                "datamodel": answer,
                "answer": answer.answer,
                "question": question,
                "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
                "retrieved_context": retrieved_context["retrieved_context"],
                "similarity_scores": retrieved_context["similarity_scores"]
        }

        return final_result

In [51]:
output = rag_pipeline("Can I get a charging cable? Please suggest a good one")

In [54]:
output

{'datamodel': RAGGenerationResponse(answer="Yes, we have excellent charging cable options available. I recommend the **PEAPOLET iPhone Charger Cable (ID: B09NCXYHMV)** with a rating of 4.4 stars. It's a 3-pack of 3.3FT Apple MFi Certified USB-A to Lightning cables that support fast charging (2.4A) and fast data transfer (480Mbps). They're durable with 10,000+ bend lifespan and compatible with iPhone 14/13/12/11 and other Apple devices.\n\nAlternatively, if you need a USB Type-C charging solution, the **Arae USB Type C to 3.5mm Headphone and Charger Adapter (ID: B0BCKCJQPN)** with a 4.4-star rating is a great 2-in-1 option that supports up to 60W PD fast charging while allowing you to listen to music simultaneously. It's compatible with Samsung Galaxy, iPad Pro, MacBook, Pixel, and other USB-C devices.", reasoning="The user asked for a charging cable recommendation. Based on the available products, there are two main charging cable options: 1) The PEAPOLET iPhone Charger Cable (B09NCXYH